# Stanford 30-class pretraining and transfer fine-tuning on AutoDL


In [ ]:
from pathlib import Path
import shutil

# raman.zip 固定放在 AutoDL 的 /root。
PROJECT_ROOT = Path.cwd()
RAMAN_ZIP_PATH = Path("/root/raman.zip")
RAMAN_DIR = PROJECT_ROOT / "raman"

RUN_UNPACK_RAMAN = RAMAN_ZIP_PATH.is_file()
CLEAR_RAMAN_BEFORE_UNPACK = True

def unpack_library(zip_path, target_dir, should_unpack, clear_before_unpack):
    if not should_unpack:
        raise FileNotFoundError(
            "找不到 raman.zip；请确认它位于 /root/raman.zip。"
        )
    if clear_before_unpack and target_dir.exists():
        shutil.rmtree(target_dir)
    shutil.unpack_archive(zip_path, PROJECT_ROOT)
    print(f"unpacked {zip_path} -> {target_dir}")

unpack_library(
    RAMAN_ZIP_PATH,
    RAMAN_DIR,
    RUN_UNPACK_RAMAN,
    CLEAR_RAMAN_BEFORE_UNPACK,
)

reference_axis_path = PROJECT_ROOT / "dataset" / "Stanforddataset" / "reference_wavenumbers.npy"
if reference_axis_path.is_file():
    print(f"reference_wavenumbers = {reference_axis_path}")
else:
    print("raman.zip 中未找到共享波数轴；后续解压 Stanford init.npz 时会自动生成。")


In [ ]:
import importlib
import sys
# 修改库代码后，先从 sys.modules 清掉 raman，再重新导入
# 这种方式能处理已经删除或改名的旧模块
def reload_all():
    for name in list(sys.modules):
        if (
            name == "raman"
            or name.startswith("raman.")
            or name == "raman_fine_tuning"
            or name.startswith("raman_fine_tuning.")
        ):
            del sys.modules[name]
    import raman.config as config_module
    print("Modules reloaded.")
    return config_module.config

config = reload_all()


In [ ]:
from pathlib import Path
import shutil
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from raman.data.profiles import get_dataset_dir, get_profile

# 在这里切换数据集，建议用英文别名。
DATASET_NAME = "Stanford"  # MICRO / GP / FUNG / Stanford
# standard：600–1800 线性 896 点；stanford_transfer：Stanford 原始 896 点共享轴。
INPUT_GRID_MODE = "stanford_transfer"
# Stanford 预训练和 Stanford→GN 微调必须同时使用 minmax。
NORM_METHOD = "minmax"

profile = get_profile(DATASET_NAME)
# 解析 dataset/ 下的目录结构。
dataset_dir = get_dataset_dir(profile, PROJECT_ROOT)
dataset_dir.mkdir(parents=True, exist_ok=True)
# dataset_root / bad_bands 会随 dataset_name 联动。网格模式必须和 build 保持一致。
config.dataset_name = DATASET_NAME
config.input_grid_mode = INPUT_GRID_MODE
config.norm_method = NORM_METHOD
pca_log_path = dataset_dir / profile.pca_log_name
cosmic_log_path = dataset_dir / profile.cosmic_ray_log_name

print("dataset_name =", config.dataset_name)
print("dataset_dir =", dataset_dir)
print("dataset_root =", config.dataset_root)
print("input_grid_mode =", config.input_grid_mode)
print("norm_method =", config.norm_method)
print("bad_bands =", config.bad_bands)
print("pca_log =", pca_log_path)
print("cosmic_ray_log =", cosmic_log_path)


In [ ]:
from raman.data.io import unpack_init

# AutoDL 上传文件通常挂载在 /root/autodl-fs；直接从该位置读取，无需复制到工作目录。
UPLOADED_STANFORD_INIT_PATH = Path("/root/autodl-fs/init.npz")
local_pack_path = dataset_dir / profile.root_init_pack
pack_path = (
    UPLOADED_STANFORD_INIT_PATH
    if UPLOADED_STANFORD_INIT_PATH.is_file()
    else local_pack_path
)
init_dir = dataset_dir / profile.root_init

# 默认离线：优先解压上传的 init.npz；只有手动改为 True 才下载公开原始数据。
DOWNLOAD_STANFORD_SOURCE = False
RUN_UNPACK_INIT = pack_path.is_file()
# 若要覆盖旧的 init，可以先清空再解压。
CLEAR_INIT_BEFORE_UNPACK = True

if DOWNLOAD_STANFORD_SOURCE:
    from raman.data.stanford import download_reference_data, import_reference_init
    download_reference_data(dataset_dir)
    import_reference_init(dataset_dir)
elif RUN_UNPACK_INIT:
    if CLEAR_INIT_BEFORE_UNPACK and init_dir.exists():
        shutil.rmtree(init_dir)
        init_dir.mkdir(parents=True, exist_ok=True)
    unpack_init(pack_path, init_dir)
else:
    raise FileNotFoundError(
        "找不到 Stanford init.npz；请确认 /root/autodl-fs/init.npz 已上传，"
        f"或放到 {local_pack_path}。"
    )

In [ ]:
from raman.data.build import build_train
from raman.pipeline import PipelineConfig

# 这一段从 init 处理到 train；独立测试集请直接放到 dataset/<数据集>/test
RUN_BUILD_TRAIN = True
# 只有想清空旧结果重跑时才打开
REBUILD_TRAIN = True

train_dir = dataset_dir / profile.root_train_clean
test_dir = dataset_dir / profile.root_test
pca_log_path = dataset_dir / profile.pca_log_name
cosmic_log_path = dataset_dir / profile.cosmic_ray_log_name

# 重建时删除旧目录；由 build_train 自己创建新目录。
# Stanford 构建要求目标 train 在开始时不存在。
def remove_dir(path):
    if path.exists():
        shutil.rmtree(path)

# build_train 直接从 init 生成 train，并执行 PCA 过滤。
# Stanford 使用迁移模式时直接选共享原始点；其他数据会插值到同一轴。
pipeline_config = PipelineConfig(input_grid_mode=INPUT_GRID_MODE)
if RUN_BUILD_TRAIN:
    if REBUILD_TRAIN:
        remove_dir(train_dir)
        # 上一次构建异常时可能留下临时目录，重建前一并清理。
        remove_dir(dataset_dir / f"{profile.root_train_clean}_building")
    build_train(profile, dataset_dir, pipeline_config=pipeline_config)

print("Done.")
print(f"train = {train_dir}")
print(f"input_grid_mode = {INPUT_GRID_MODE}")
print(f"test = {test_dir}")
print(f"pca_log = {pca_log_path}")
print(f"cosmic_ray_log = {cosmic_log_path}")


In [ ]:
from raman.data.count import count_dataset, print_results

# 默认统计 train，也可以改成 test。
COUNT_SUBDIR = "train"
target_dir = dataset_dir / COUNT_SUBDIR

tree, total_files = count_dataset(target_dir)
print_results(tree, total_files)


In [ ]:
# Stanford 预训练配置
STANFORD_PRETRAIN_LEVEL = "level_1"
TRAIN_ONLY_PARENT_NAME = None
TRAIN_ONLY_PARENT = None

OVERRIDE_ALIGN_LOSS_WEIGHT = None
OVERRIDE_SUPCON_TAU = None
OVERRIDE_SUPCON_LOSS_WEIGHT = None
OVERRIDE_SEED = None
OVERRIDE_SPLIT_BY_SOURCE_PREFIX = None

# 训练轮次和早停轮次；None 表示使用 raman.config.py 的默认值。
OVERRIDE_EPOCHS = None
OVERRIDE_PATIENCE = None

SUPCON_START_OVERRIDE = None
SUPCON_END_OVERRIDE = None
ALIGN_START_OVERRIDE = None
ALIGN_END_OVERRIDE = None
OVERRIDE_OUTPUT_DIR = None

# 第一次按顺序运行时保持 True；复用已有 Stanford run 时改为 False 并填写路径。
RUN_STANFORD_PRETRAIN = True
MANUAL_STANFORD_RUN_DIR = None  # 例如 output/Stanford/<实验>/level_1/run_<时间>


In [ ]:
from dataclasses import asdict, is_dataclass

from raman.data.profiles import apply_training_profile_defaults

# Stanford 的固定训练参数由 profile 管理；先应用再打印，避免显示全局默认值。
config = apply_training_profile_defaults(config)


def _config_group_to_dict(group):
    if group is None:
        return {}
    if is_dataclass(group):
        return asdict(group)
    if hasattr(group, "to_dict"):
        return group.to_dict()
    return {
        key: value
        for key, value in vars(group).items()
        if not key.startswith("_")
    }


def _print_config_section(title, data):
    print(f"\n===== {title} =====")
    if not data:
        print("  <empty>")
        return
    for key in data:
        print(f"  {key}: {data[key]}")


_print_config_section(
    "Derived Values",
    {
        "dataset_root": getattr(config, "dataset_root", None),
        "in_channels": getattr(config, "in_channels", None),
        "delta": getattr(config, "delta", None),
    },
)
_print_config_section("Shared Input Config", _config_group_to_dict(getattr(config, "shared", None)))
_print_config_section("Model Run Config", _config_group_to_dict(getattr(config, "model", None)))
_print_config_section("Runtime Config", _config_group_to_dict(getattr(config, "runtime", None)))
_print_config_section(
    "Level Settings",
    {
        "train_per_parent": getattr(config, "train_per_parent", False),
        "use_align_loss": f"{getattr(config, 'use_align_loss', True)} (weight={getattr(config, 'align_loss_weight', 0.05)})",
        "use_supcon_loss": f"{getattr(config, 'use_supcon_loss', True)} (weight={getattr(config, 'supcon_loss_weight', 0.03)}, tau={getattr(config, 'supcon_tau', 0.15)})",
    },
)


In [ ]:
from raman.trainer import TrainOverrides, run_training

# 第一段：训练 Stanford 30 类预训练模型。
if RUN_STANFORD_PRETRAIN:
    stanford_train_result = run_training(
        config,
        overrides=TrainOverrides(
            current_train_level=STANFORD_PRETRAIN_LEVEL,
            train_only_parent_name=TRAIN_ONLY_PARENT_NAME,
            train_only_parent=TRAIN_ONLY_PARENT,
            override_align_loss_weight=OVERRIDE_ALIGN_LOSS_WEIGHT,
            override_supcon_tau=OVERRIDE_SUPCON_TAU,
            override_supcon_loss_weight=OVERRIDE_SUPCON_LOSS_WEIGHT,
            override_output_dir=OVERRIDE_OUTPUT_DIR,
        ),
    )
    STANFORD_PRETRAIN_EXP_DIR = stanford_train_result["output_dir"]
    stanford_run_dirs = stanford_train_result.get("run_dirs", [])
    if len(stanford_run_dirs) != 1:
        raise RuntimeError(f"预训练期望得到一个 run，实际为：{stanford_run_dirs}")
    STANFORD_PRETRAIN_RUN_DIR = str(
        (Path(STANFORD_PRETRAIN_EXP_DIR) / stanford_run_dirs[0]).resolve()
    )
else:
    if MANUAL_STANFORD_RUN_DIR is None:
        raise ValueError("跳过 Stanford 训练时，必须填写 MANUAL_STANFORD_RUN_DIR")
    STANFORD_PRETRAIN_RUN_DIR = str(Path(MANUAL_STANFORD_RUN_DIR).resolve())
    STANFORD_PRETRAIN_EXP_DIR = str(Path(STANFORD_PRETRAIN_RUN_DIR).parents[1])

EXP_DIR = STANFORD_PRETRAIN_EXP_DIR
print("STANFORD_PRETRAIN_EXP_DIR =", STANFORD_PRETRAIN_EXP_DIR)
print("STANFORD_PRETRAIN_RUN_DIR =", STANFORD_PRETRAIN_RUN_DIR)


## 结果通用配置


In [ ]:
# 结果入口与 notebook 默认参数；仅在需要覆盖默认值时填写 NOTEBOOK_OPTIONS。
from raman.tool.notebook import NotebookTools

MANUAL_EXP_DIR = None
if MANUAL_EXP_DIR is not None:
    EXP_DIR = MANUAL_EXP_DIR
elif "EXP_DIR" not in globals():
    EXP_DIR = None

NOTEBOOK_OPTIONS = {}
notebook_tools = NotebookTools(**NOTEBOOK_OPTIONS)

if EXP_DIR is not None:
    print("EXP_DIR =", EXP_DIR)
    notebook_tools.list_run_slots(EXP_DIR)
notebook_tools.clear_run_selection()


## 单模型检查


In [ ]:
# 第二段：先检查 Stanford 预训练验证集结果。
SINGLE_RUN_DIR = STANFORD_PRETRAIN_RUN_DIR
SINGLE_LEVEL = STANFORD_PRETRAIN_LEVEL
SINGLE_PARENT_IDX = None

notebook_tools.clear_run_selection()
LAST_RESULT_DIR = SINGLE_RUN_DIR
print("Stanford pretrain run =", SINGLE_RUN_DIR)


In [ ]:
# Stanford 预训练验证集结果和混淆矩阵。
from raman.eval import run_eval_single_model

stanford_val_result_dir = run_eval_single_model(
    SINGLE_RUN_DIR,
    level=SINGLE_LEVEL,
)
LAST_RESULT_DIR = stanford_val_result_dir
print("stanford_val_result_dir =", stanford_val_result_dir)
notebook_tools.show_confusion_matrix(stanford_val_result_dir)


In [ ]:
# 第三段：设置 Stanford → GN 微调参数；微调专属配置不写入通用 raman.config。
from raman_fine_tuning import FineTuneConfig, print_fine_tune_plan

UNFREEZE_TAIL = True
WARM_START_RUN_DIR = None  # 若从已有 GN 微调结果续接，填写对应 run_* 路径。
FINE_TUNE_CONFIG = FineTuneConfig(
    source_run_dir=STANFORD_PRETRAIN_RUN_DIR,
    target_dataset_name="GN",
    learning_rate=5e-5,
    unfreeze_tail=UNFREEZE_TAIL,
    warm_start_run_dir=WARM_START_RUN_DIR,
)
fine_tune_target_config = print_fine_tune_plan(config, FINE_TUNE_CONFIG)


In [ ]:
# 第四段：准备目标数据；已有 init/ 与 train/ 时直接复用。
from raman_fine_tuning import prepare_target_dataset

REBUILD_TARGET_TRAIN = False  # 更换 init.npz 或需按共享波数轴重建时才改为 True。
target_context = prepare_target_dataset(
    PROJECT_ROOT,
    FINE_TUNE_CONFIG,
    rebuild_train=REBUILD_TARGET_TRAIN,
)
profile = target_context.profile
dataset_dir = target_context.dataset_dir
train_dir = target_context.train_dir


In [ ]:
# 第五段：执行微调并记录生成的目标实验目录。
from raman_fine_tuning import run_single_fine_tuning

fine_tune_result, FINE_TUNE_RUN_DIR = run_single_fine_tuning(
    config,
    FINE_TUNE_CONFIG,
)
FINE_TUNE_EXP_DIR = fine_tune_result["output_dir"]
EXP_DIR = FINE_TUNE_EXP_DIR
print("FINE_TUNE_EXP_DIR =", FINE_TUNE_EXP_DIR)
print("FINE_TUNE_RUN_DIR =", FINE_TUNE_RUN_DIR)


In [ ]:
# 第五段：输出微调后的目标数据集验证结果。
SINGLE_RUN_DIR = FINE_TUNE_RUN_DIR
SINGLE_LEVEL = FINE_TUNE_LEVEL
SINGLE_PARENT_IDX = FINE_TUNE_PARENT

fine_tune_val_result_dir = run_eval_single_model(
    SINGLE_RUN_DIR,
    level=SINGLE_LEVEL,
)
LAST_RESULT_DIR = fine_tune_val_result_dir
print("fine_tune_val_result_dir =", fine_tune_val_result_dir)
notebook_tools.show_confusion_matrix(fine_tune_val_result_dir)


In [ ]:
# 单模型 baseline 计算和混淆矩阵展示
from raman.eval import run_baseline_single_model

single_baseline_result_dir = run_baseline_single_model(
    SINGLE_RUN_DIR,
    level=SINGLE_LEVEL,
    **notebook_tools.baseline_kwargs(),
)
LAST_RESULT_DIR = single_baseline_result_dir
print("single_baseline_result_dir =", single_baseline_result_dir)
notebook_tools.show_confusion_matrix(single_baseline_result_dir)


In [ ]:
# 单模型 analysis 计算和图展示
from raman.analysis import run_analysis_single_model

single_analysis_result_dir = run_analysis_single_model(
    SINGLE_RUN_DIR,
    level=SINGLE_LEVEL,
    parent_idx=SINGLE_PARENT_IDX,
    inherit_missing_levels=notebook_tools.inherit_missing_levels,
    heatmap_cfg=notebook_tools.heatmap_config(),
)
LAST_RESULT_DIR = single_analysis_result_dir
print("single_analysis_result_dir =", single_analysis_result_dir)
notebook_tools.show_analysis_figures(single_analysis_result_dir)


In [ ]:
# 最后压缩整个实验目录
# 日常局部下载可使用前面的单 run 或选择目录打包单元；这里用于完整保存 EXP_DIR
FINAL_PACKAGE_EXP_DIR = EXP_DIR
FINAL_PACKAGE_OUTPUT_DIR = "/content"
FINAL_PACKAGE_NAME = None

if FINAL_PACKAGE_EXP_DIR is None:
    raise ValueError("请填写 EXP_DIR，或先运行训练得到实验目录")

final_zip = notebook_tools.package_directory(
    FINAL_PACKAGE_EXP_DIR,
    output_dir=FINAL_PACKAGE_OUTPUT_DIR,
    package_name=FINAL_PACKAGE_NAME,
)


In [ ]:
# 最后将当前数据集的 train 与 init 一起压缩到 data.zip
import zipfile

DATA_PACKAGE_SOURCE_DIRS = [train_dir, dataset_dir / profile.root_init]
DATA_PACKAGE_PATH = Path("/content/data.zip")

with zipfile.ZipFile(DATA_PACKAGE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for source_dir in DATA_PACKAGE_SOURCE_DIRS:
        if not source_dir.is_dir():
            raise FileNotFoundError(f"找不到要压缩的目录：{source_dir}")
        for path in sorted(source_dir.rglob("*")):
            if path.is_file():
                archive.write(path, arcname=path.relative_to(source_dir.parent))

print("package_sources =", DATA_PACKAGE_SOURCE_DIRS)
print("package_zip =", DATA_PACKAGE_PATH)
